In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import xarray as xr
import glob
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------
SILO_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/combined_climate"
OUT_DIR  = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/combined_climate_growing_season/Climate_variables"
os.makedirs(OUT_DIR, exist_ok=True)

# Climate variables to keep
CLIMATE_VARS = [
    "monthly_rain",
    "max_temp",
    "min_temp",
    "radiation",
    "vp",
    "vp_deficit",
    "rh_tmax",
    "rh_tmin"
]

# Growing season months (May–October)
GROW_MONTHS = [5, 6, 7, 8, 9, 10]

# --------------------------------------------------
# Process each yearly SILO file
# --------------------------------------------------
silo_files = sorted(glob.glob(f"{SILO_DIR}/*.nc"))

for f in silo_files:
    ds = xr.open_dataset(f)

    year = ds.time.dt.year.values[0]

    # Select growing season months
    ds_grow = ds.sel(
        time=ds.time.dt.month.isin(GROW_MONTHS)
    )[CLIMATE_VARS]

    # Sanity check
    assert ds_grow.dims["time"] == 6, f"Wrong months in {year}"

    # Save
    out_file = f"{OUT_DIR}/silo_growing_season_{year}.nc"
    ds_grow.to_netcdf(out_file)

    print(f"✅ Saved growing season climate for {year}")


In [ ]:
import xarray as xr

ds = xr.open_dataset(r"Data\raster\combined_climate_growing_season/silo_growing_season_2000.nc")
print(ds)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd

# === Load the shapefile for Queensland regions ===
shapefile_path = 'Eastern_Aus_Bunbury\Eastern_Aus_Bunbury.shp'
qld_shape = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

# === Open the monthly max temperature data for QLD ===
ds = xr.open_dataset(r'Data\raster\combined_climate_growing_season\silo_growing_season_1991.nc')
monthly_rain = ds['monthly_rain']  # Adjust variable name as needed

# === Create figure and axes for the plot ===
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(20, 12))
axes = axes.flatten()

# === Plot each month with its own colorbar ===
for i, month in enumerate(monthly_rain.time):
    ax = axes[i]
    data = monthly_rain.sel(time=month)

    # Plot with individual colorbar
    data.plot(ax=ax, cmap='cividis', add_colorbar=True)

    # Overlay region boundaries
    qld_shape.boundary.plot(ax=ax, color='black', linewidth=0.5, alpha=0.7)

    # Set the title
    ax.set_title(month.dt.strftime('%B').item())
    ax.set_xlabel('')
    ax.set_ylabel('')

# === Remove unused axes if any ===
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

# === Final layout ===
plt.tight_layout()
plt.show()


Drought growing monthsn

In [ ]:
import xarray as xr
import glob
import os

# --------------------------------------------------
# Paths
# --------------------------------------------------
SILO_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/combined_drought_indices"
OUT_DIR  = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/combined_climate_growing_season/Drought_variables"
os.makedirs(OUT_DIR, exist_ok=True)

# Climate variables to keep
CLIMATE_VARS = [
    "spi_1",
    "spi_3",
    "spi_6",
    "spi_12",
    "spei_1",
    "spei_3",
    "spei_6",
    "spei_12"
]

# Growing season months (May–October)
GROW_MONTHS = [5, 6, 7, 8, 9, 10]

# --------------------------------------------------
# Process each yearly SILO file
# --------------------------------------------------
silo_files = sorted(glob.glob(f"{SILO_DIR}/*.nc"))

for f in silo_files:
    ds = xr.open_dataset(f)

    year = ds.time.dt.year.values[0]

    # Select growing season months
    ds_grow = ds.sel(
        time=ds.time.dt.month.isin(GROW_MONTHS)
    )[CLIMATE_VARS]

    # Sanity check
    assert ds_grow.dims["time"] == 6, f"Wrong months in {year}"

    # Save
    out_file = f"{OUT_DIR}/drought_growing_season_{year}.nc"
    ds_grow.to_netcdf(out_file)

    print(f"✅ Saved growing season climate for {year}")


In [ ]:
import xarray as xr

ds = xr.open_dataset(r"/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/combined_climate_growing_season/Drought_variables/drought_growing_season_1991.nc")
print(ds)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

# === Load the shapefile ===
shapefile_path = r'/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Eastern_Aus_Bunbury/Eastern_Aus_Bunbury.shp'
qld_shape = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

# === Open the SPI dataset ===
nc_path = Path(r'/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/combined_climate_growing_season/Drought_variables/drought_growing_season_1991.nc')
year = nc_path.stem.split('_')[1]

ds = xr.open_dataset(nc_path)

# === Select SPI variable ONCE ===
var_name = 'spei_1'          # change here: spi_3, spi_6, etc.
monthly_rain = ds[var_name]

# === Convert variable name to display label ===
spi_label = var_name.replace('_', '-').upper()   # spi_1 → SPI-1

# === Fix metadata so colorbar label is correct ===
monthly_rain.attrs.pop('units', None)
monthly_rain.attrs['long_name'] = spi_label

# === Create figure and axes ===
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 12))
axes = axes.flatten()

# === Plot each month ===
for i, month in enumerate(monthly_rain.time):
    ax = axes[i]
    data = monthly_rain.sel(time=month)

    data.plot(
        ax=ax,
        cmap='RdBu',
        add_colorbar=True
    )

    # Overlay region boundaries
    qld_shape.boundary.plot(
        ax=ax,
        color='black',
        linewidth=0.5,
        alpha=0.7
    )

    # Titles and formatting
    ax.set_title(month.dt.strftime('%B').item(), fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')

# === Remove unused subplots ===
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# === Figure title (automatic) ===
plt.suptitle(
    f'Monthly {spi_label} for {year} in WheatBelt',
    fontsize=16,
    y=0.98
)

plt.tight_layout()
plt.show()
